In [13]:
import os
import time
import tempfile

# Create a mock PD0 file for benchmarking
def create_mock_pd0_file(num_ensembles, ensemble_size):
    """
    Create a mock PD0 file with a given number of ensembles.
    Each ensemble starts with 0x7f 0x7f and has a fixed-size body with a 2-byte checksum.
    """
    SYNC_BYTES = bytes([0x7f, 0x7f])
    CHECKSUM = bytes([0x00, 0x00])  # Dummy checksum
    body_size = ensemble_size - 4  # exclude 2 sync bytes and 2-byte size field
    ensemble_data = bytearray()

    for _ in range(num_ensembles):
        # Construct ensemble: header + body + checksum
        size = ensemble_size - 2  # PD0 size excludes checksum
        size_bytes = bytes([size & 0xFF, (size >> 8) & 0xFF])
        header = SYNC_BYTES + size_bytes + bytes([0] * (ensemble_size - 4))
        ensemble = header + CHECKSUM
        ensemble_data += ensemble

    # Write to temporary file
    temp_file = tempfile.NamedTemporaryFile(delete=False)
    temp_file.write(ensemble_data)
    temp_file.close()
    return temp_file.name

# Define both index-finding methods for benchmarking
def streaming_index_finder(file_path):
    # Open file
    file = open(file_path, "rb")

    # Variable for keeping track of ensemble positions, list for ensemble indexes
    current_offset = 0
    ens_indexes = []

    # Search the entire file for ensemble headers
    while True:
        # Get header ID, source ID, and numbytes
        header = file.read(4)

        # End of file check
        if len(header) < 4:
            break

        # If the two start bytes are right (two 7f bytes in a row)
        if header[0] == 0x7f and header[1] == 0x7f:
            # Get ensemble size
            ens_size = header[2] + (header[3] << 8) + 2

            # If size is within expected bounds (removes potential errors from random 7f7f data)
            if 32 <= ens_size <= 4096:
                ens_indexes.append(current_offset)
                current_offset += ens_size
                file.seek(current_offset)
                continue
            else:
                # Continue seeking
                current_offset += 1
                file.seek(current_offset)
        else:
            # Continue seeking
            current_offset += 1
            file.seek(current_offset)
    return ens_indexes

def array_scan_index_finder(file_path):
    import numpy as np

    file = open(file_path, "rb")

    # Get first data batch
    buffer_size = int(1e6)
    eoe = 127

    # Get the end of the file (needed for when file is larger than buffer size)
    file.seek(0, 2)
    eof = file.tell()
    file.seek(0, 0)  # Reset position

    # Read a batch of data
    batch = np.fromfile(file, dtype="uint8", count=buffer_size)

    # Appending with numpy arrays is inefficient so use a list
    increment = list(np.fromfile(file, dtype="uint8", count=2))

    # Search until the next ensemble starts so nothing gets split
    while (len(increment) < 2 or (increment[-1] != eoe and increment[-2] != eoe)) and (file.tell() != eof):
        # Read the next byte
        next_byte = file.read(1)

        # If byte is empty, break from loop
        if not next_byte:
            break
        increment.append(next_byte[0])

    # Convert increment to ndarray and add to data
    increment = np.array(increment, dtype="uint8")
    data = np.concatenate((batch, increment[:-1]))

    # List for storing start index of each ensamble
    ens_indexes = []

    # Offset for later batches
    cumulative_index = 0

    # Search entire file for ensemble headers
    while data.size > 0:
        # Get index of all 7f values in the data
        potentialIndexes = np.where(data == 0x7f)[0]

        # Check find where there are two consecutive 7f values indicating an ensemble header
        headers = (np.diff(potentialIndexes) == 1)
        headers = np.append(headers, False)

        # Get index of all headers
        headerIndexes = potentialIndexes[headers]
        headerIndexes += cumulative_index

        # Add header indexes to list, update offset
        ens_indexes.extend(headerIndexes.tolist())
        cumulative_index += data.size

        #print(f"{len(data)} bytes scanned with {len(headerIndexes)} ensembles.")

        # Check the rest of the file by updating data
        # Read a batch of data
        batch = np.fromfile(file, dtype="uint8", count=buffer_size)

        # Appending with numpy arrays is inefficient so use a list
        increment = list(np.fromfile(file, dtype="uint8", count=2))

        # Search until the next ensemble starts so nothing gets split
        while (len(increment) < 2 or (increment[-1] != eoe and increment[-2] != eoe)) and (file.tell() != eof):
            # Read the next byte
            next_byte = file.read(1)

            # If byte is empty, break from loop
            if not next_byte:
                break
            increment.append(next_byte[0])

        # Convert increment to ndarray and add to data
        increment = np.array(increment, dtype="uint8")
        data = np.concatenate((batch, increment[:-1]))

    return ens_indexes

# Run benchmark
mock_file_path = create_mock_pd0_file(2629800, 2232)

print("file made")

start_time = time.time()
streaming_indexes = streaming_index_finder(mock_file_path)
streaming_duration = time.time() - start_time

print("stream done")

start_time = time.time()
scan_indexes = array_scan_index_finder(mock_file_path)
scan_duration = time.time() - start_time

# Clean up
os.remove(mock_file_path)

(streaming_duration, scan_duration, len(streaming_indexes), len(scan_indexes))

file made
stream done


(3.2947123050689697, 4.5519936084747314, 2629800, 2623930)

In [12]:
import gc
gc.collect()

0